# Tugas 5 | Ekstraksi - LDA

Tugas 5 melakuakan Ekstrkasi Fitur menggunakan **Latent Dirichlet Allocation (LDA).**

Kolom yang digunakan:
- clean_stemmed (fitur text)
- Kategori (label)

## LDA

In [1]:
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# Baca dataset
df = pd.read_csv("detik_cleaned.csv")

# Pastikan kolom yang diperlukan ada
required_cols = {"clean_stemmed", "Kategori"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"Dataset harus punya kolom: {required_cols}")

# Ambil kolom teks yang sudah dibersihkan
texts = df["clean_stemmed"].dropna().astype(str)

# Ubah teks ke bentuk Bag of Words
# LDA tidak bisa langsung pakai teks mentah, harus pakai CountVectorizer
vectorizer = CountVectorizer(
    max_df=0.95, # abaikan kata yang muncul di >95% dokumen
    min_df=2, # abaikan kata yang muncul <2 dokumen
    stop_words=None # karena teks kamu sudah bersih
)
X = vectorizer.fit_transform(texts)

print(f"Jumlah dokumen: {X.shape[0]}, Jumlah kata unik: {X.shape[1]}")

# Inisialisasi dan latih model LDA
n_topics = 10
lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method='batch'
)

lda_features = lda_model.fit_transform(X)  # menghasilkan distribusi topik untuk tiap dokumen

# Konversi hasil LDA ke DataFrame
lda_df = pd.DataFrame(
    lda_features,
    columns=[f"Topic_{i+1}" for i in range(n_topics)]
)

# Tambahkan kolom kategori supaya siap untuk klasifikasi
lda_df.insert(0, "Kategori", df.loc[texts.index, "Kategori"].values)

# Simpan ke CSV
lda_df.to_csv("lda_result.csv", index=False)
print("✅ Ekstraksi fitur LDA berhasil disimpan sebagai lda_result.csv")

# (Opsional) Tampilkan kata paling dominan di tiap topik
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\n🟩 Topik {topic_idx + 1}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]))

print_top_words(lda_model, vectorizer.get_feature_names_out())


Jumlah dokumen: 100, Jumlah kata unik: 1342
✅ Ekstraksi fitur LDA berhasil disimpan sebagai lda_result.csv

🟩 Topik 1:
latih anda ada indonesia para isi alat air pelatnas di

🟩 Topik 2:
hidup jelas di hanya yang milik kita sama cara bahasa

🟩 Topik 3:
air apa jakarta main pertama di proliga putri 2024 voli

🟩 Topik 4:
guna di alat milik orang temu cara itu hasil sehat

🟩 Topik 5:
suara motogp guna marquez balap musik bagnaia menang martin foto

🟩 Topik 6:
jauh guna balap apa salah ilmuwan langkah memang di temu

🟩 Topik 7:
lokasi kerja tempat di sehat media kontrak posisi wilayah apa

🟩 Topik 8:
guru 2024 didik guna di mata sistem itu apa ajar

🟩 Topik 9:
2025 januari air sekolah mudah kita bisa anak awal nomor

🟩 Topik 10:
barang pesawat buku air mahasiswa guna ada hidup sebab dapat
